# Universidad Mayor

## Métodos de Inteligencia Artificial

# Trabajo de Desarrollo Individual

## Clasificación Binaria del Precio de Viviendas mediante Machine Learning

---

**Alumno:** Pablo Maldonado Silva

**Dataset:** datos.csv

---

### Objetivo

Desarrollar distintos modelos de Machine Learning supervisado para clasificar viviendas según su precio de venta utilizando técnicas de clasificación binaria.

# Introducción

En este trabajo se utilizará un conjunto de datos correspondiente al mercado inmobiliario.

La variable objetivo original es **SalePrice**, que representa el precio de venta de cada vivienda.

Como la actividad solicita resolver un problema de **clasificación binaria**, esta variable será transformada en dos categorías:

- Precio Bajo (0)
- Precio Alto (1)

Posteriormente se entrenarán cuatro modelos:

- Regresión Logística con regularización Ridge (L2)
- Random Forest
- XGBoost
- Red Neuronal (MLP)

Finalmente se compararán utilizando Accuracy, Precision, Recall, F1-Score, AUC y KS.

In [ ]:
# ==========================================
# IMPORTACIÓN DE LIBRERÍAS
# ==========================================

import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

plt.style.use("ggplot")

In [ ]:
# ==========================================
# CARGA DEL DATASET
# ==========================================

df = pd.read_csv("datos.csv")

In [ ]:
from google.colab import files
import io

print('Por favor, selecciona el archivo datos.csv para cargarlo:')
uploaded = files.upload()

for fn in uploaded.keys():
  print(f'Archivo "{fn}" cargado.')

Por favor, selecciona el archivo datos.csv para cargarlo:


In [ ]:
df.head()

In [ ]:
print("Número de filas:", df.shape[0])
print("Número de columnas:", df.shape[1])

In [ ]:
df.info()

In [ ]:
df.describe().T

In [ ]:
df.isnull().sum()

In [ ]:
plt.figure(figsize=(12,5))

sns.heatmap(df.isnull(), cmap="viridis", cbar=False)

plt.title("Mapa de valores faltantes")

plt.show()

In [ ]:
df.hist(figsize=(15,12))

plt.tight_layout()

plt.show()

In [ ]:
plt.figure(figsize=(8,5))

sns.histplot(df["SalePrice"], bins=30, kde=True)

plt.title("Distribución del Precio de Venta")

plt.xlabel("SalePrice")

plt.ylabel("Frecuencia")

plt.show()

In [ ]:
corr = df.corr(numeric_only=True)

plt.figure(figsize=(12,10))

sns.heatmap(
    corr,
    cmap="coolwarm",
    annot=False
)

plt.title("Matriz de Correlación")

plt.show()

In [ ]:
corr["SalePrice"].sort_values(ascending=False)

In [ ]:
plt.figure(figsize=(8,5))

sns.scatterplot(
    data=df,
    x="GrLivArea",
    y="SalePrice"
)

plt.title("Área Habitable vs Precio")

plt.show()

In [ ]:
plt.figure(figsize=(8,5))

sns.scatterplot(
    data=df,
    x="GarageArea",
    y="SalePrice"
)

plt.title("Área del Garaje vs Precio")

plt.show()

In [ ]:
plt.figure(figsize=(8,5))

sns.scatterplot(
    data=df,
    x="TotalBsmtSF",
    y="SalePrice"
)

plt.title("Superficie del Sótano vs Precio")

plt.show()

In [ ]:
plt.figure(figsize=(8,5))

sns.boxplot(
    data=df,
    x="GarageCars",
    y="SalePrice"
)

plt.title("Cantidad de Garajes vs Precio")

plt.show()

In [ ]:
df["MSZoning"].value_counts()

In [ ]:
plt.figure(figsize=(7,5))

sns.countplot(
    data=df,
    x="MSZoning"
)

plt.title("Distribución por Zona")

plt.show()

# Conclusiones del análisis exploratorio

El conjunto de datos contiene 1460 registros y 20 variables, incluyendo variables numéricas y categóricas. La exploración muestra que características como el área habitable (`GrLivArea`), el tamaño del sótano (`TotalBsmtSF`) y la capacidad del garaje (`GarageCars`) presentan una relación positiva con el precio de venta. También se identificó la presencia de algunas variables categóricas que requerirán codificación antes del entrenamiento de los modelos.

En la siguiente sección se prepararán los datos para el proceso de clasificación binaria.

# Preparación de los datos

Antes de entrenar los modelos de Machine Learning es necesario preparar los datos. En esta etapa se realizará la creación de la variable objetivo binaria, el tratamiento de valores faltantes, la codificación de variables categóricas y la normalización de los datos cuando sea necesario.

In [ ]:
# ==========================================
# CREACIÓN DE LA VARIABLE OBJETIVO
# ==========================================

mediana = df["SalePrice"].median()

print("Mediana del precio:", mediana)

In [ ]:
df["Precio_Alto"] = (df["SalePrice"] > mediana).astype(int)

df[["SalePrice", "Precio_Alto"]].head()

In [ ]:
df["Precio_Alto"].value_counts()

In [ ]:
plt.figure(figsize=(6,4))

sns.countplot(data=df, x="Precio_Alto")

plt.title("Distribución de la Variable Objetivo")

plt.xlabel("Clase")

plt.ylabel("Cantidad")

plt.show()

## Eliminación de la variable objetivo original

Una vez creada la variable **Precio_Alto**, la variable continua **SalePrice** deja de ser necesaria y se elimina para evitar que el modelo tenga acceso directo al valor que intenta clasificar.

In [ ]:
X = df.drop(columns=["SalePrice", "Precio_Alto"])

y = df["Precio_Alto"]

In [ ]:
print("Variables predictoras:", X.shape)

print("Variable objetivo:", y.shape)

## Codificación de variables categóricas

Los algoritmos de Machine Learning requieren trabajar con variables numéricas. Por ello, las variables categóricas se transformarán utilizando la técnica **One-Hot Encoding**.

In [ ]:
X = pd.get_dummies(
    X,
    drop_first=True
)

In [ ]:
print("Nuevo número de variables:")

X.shape

In [ ]:
X.head()

## División del conjunto de datos

El conjunto de datos será dividido en entrenamiento y prueba.

- 70 % entrenamiento
- 30 % prueba

Además se utilizará **stratify** para mantener el equilibrio entre ambas clases.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(

    X,

    y,

    test_size=0.30,

    random_state=42,

    stratify=y

)

In [ ]:
print("Entrenamiento")

print(X_train.shape)

print()

print("Prueba")

print(X_test.shape)

## Escalamiento de los datos

Algunos algoritmos, como la Regresión Logística y las Redes Neuronales, son sensibles a la escala de las variables. Por ello, se aplicará una estandarización utilizando **StandardScaler**.

In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)

X_test_scaled = scaler.transform(X_test)

In [ ]:
print(type(X_train_scaled))

print(X_train_scaled.shape)

In [ ]:
pd.DataFrame(X_train_scaled).head()

# Conclusiones de la preparación de datos

En esta etapa se preparó el conjunto de datos para el entrenamiento de los modelos. Se creó una nueva variable objetivo binaria denominada **Precio_Alto**, se eliminaron las variables que podrían generar fuga de información, se transformaron las variables categóricas mediante One-Hot Encoding y se dividieron los datos en conjuntos de entrenamiento y prueba. Finalmente, se aplicó un proceso de estandarización para los modelos que requieren variables en una misma escala.

Con los datos preparados, el siguiente paso consiste en entrenar y comparar distintos algoritmos de Machine Learning para determinar cuál ofrece el mejor desempeño en la clasificación de viviendas según su precio.

# Entrenamiento de los modelos

En esta sección se entrenarán cuatro algoritmos de clasificación supervisada para comparar su capacidad predictiva.

Los modelos evaluados serán:

- Regresión Logística con regularización L2 (Ridge)
- Random Forest
- XGBoost
- Red Neuronal (MLP)

Cada uno será evaluado utilizando Accuracy, Precision, Recall, F1-Score y AUC.

In [ ]:
# ==========================================
# LIBRERÍAS PARA LOS MODELOS
# ==========================================

from sklearn.linear_model import LogisticRegression

from sklearn.ensemble import RandomForestClassifier

from sklearn.neural_network import MLPClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_curve
)

In [ ]:
# Instalar XGBoost (solo la primera vez en Colab)

!pip -q install xgboost

In [ ]:
from xgboost import XGBClassifier

In [ ]:
# ==========================================
# FUNCIÓN DE EVALUACIÓN
# ==========================================

def evaluar_modelo(nombre, modelo, Xtr, Xts):

    modelo.fit(Xtr, y_train)

    pred = modelo.predict(Xts)

    proba = modelo.predict_proba(Xts)[:,1]

    resultados = {

        "Modelo": nombre,

        "Accuracy": accuracy_score(y_test, pred),

        "Precision": precision_score(y_test, pred),

        "Recall": recall_score(y_test, pred),

        "F1": f1_score(y_test, pred),

        "AUC": roc_auc_score(y_test, proba)

    }

    return resultados

## Modelo 1: Regresión Logística (Ridge)

Se utiliza regularización L2 para disminuir el sobreajuste y mejorar la capacidad de generalización del modelo.

In [ ]:
ridge = LogisticRegression(

    penalty="l2",

    max_iter=5000,

    random_state=42

)

resultado_ridge = evaluar_modelo(

    "Ridge",

    ridge,

    X_train_scaled,

    X_test_scaled

)

resultado_ridge

## Modelo 2: Random Forest

Random Forest construye múltiples árboles de decisión y combina sus resultados mediante votación para mejorar la precisión.

In [ ]:
rf = RandomForestClassifier(

    random_state=42

)

resultado_rf = evaluar_modelo(

    "Random Forest",

    rf,

    X_train,

    X_test

)

resultado_rf

## Modelo 3: XGBoost

XGBoost es un algoritmo de Gradient Boosting ampliamente utilizado por su alto rendimiento en problemas de clasificación.

In [ ]:
xgb = XGBClassifier(

    random_state=42,

    eval_metric="logloss"

)

resultado_xgb = evaluar_modelo(

    "XGBoost",

    xgb,

    X_train,

    X_test

)

resultado_xgb

## Modelo 4: Red Neuronal

Se entrenará una red neuronal multicapa (MLP) con una capa oculta de 100 neuronas y función de activación ReLU.

In [ ]:
mlp = MLPClassifier(

    hidden_layer_sizes=(100,),

    activation="relu",

    max_iter=1000,

    random_state=42

)

resultado_mlp = evaluar_modelo(

    "MLP",

    mlp,

    X_train_scaled,

    X_test_scaled

)

resultado_mlp

# Comparación de modelos

A continuación se construye una tabla resumen con las principales métricas obtenidas por cada algoritmo.

In [ ]:
resultados = pd.DataFrame([

    resultado_ridge,

    resultado_rf,

    resultado_xgb,

    resultado_mlp

])

resultados

In [ ]:
resultados.sort_values(

    by="AUC",

    ascending=False

)

In [ ]:
plt.figure(figsize=(10,5))

sns.barplot(

    data=resultados,

    x="Modelo",

    y="Accuracy"

)

plt.title("Comparación de Accuracy")

plt.show()

In [ ]:
plt.figure(figsize=(10,5))

sns.barplot(

    data=resultados,

    x="Modelo",

    y="AUC"

)

plt.title("Comparación del AUC")

plt.show()

# Interpretación de resultados

Los resultados obtenidos permiten comparar el desempeño de los cuatro modelos de clasificación.

Las métricas Accuracy, Precision, Recall, F1-Score y AUC entregan una visión integral del comportamiento de cada algoritmo, facilitando la identificación del modelo con mejor capacidad predictiva.

En la siguiente sección se optimizarán los hiperparámetros mediante **GridSearchCV** y se calculará el estadístico **Kolmogorov-Smirnov (KS)** para seleccionar el modelo final.

# Optimización de hiperparámetros

Una vez comparados los modelos base, se optimizará el algoritmo con mejor desempeño utilizando **GridSearchCV**, buscando la combinación de hiperparámetros que maximice el AUC durante la validación cruzada.

In [ ]:
from sklearn.model_selection import GridSearchCV

In [ ]:
parametros = {

    "n_estimators":[100,200],

    "max_depth":[3,5,7],

    "learning_rate":[0.01,0.1],

    "subsample":[0.8,1]

}

In [ ]:
grid = GridSearchCV(

    estimator=XGBClassifier(
        random_state=42,
        eval_metric="logloss"
    ),

    param_grid=parametros,

    scoring="roc_auc",

    cv=5,

    n_jobs=-1

)

grid.fit(X_train,y_train)

In [ ]:
print("Mejores parámetros")

grid.best_params_

In [ ]:
print("Mejor AUC obtenido")

grid.best_score_

# Evaluación del modelo optimizado

In [ ]:
modelo = grid.best_estimator_

pred = modelo.predict(X_test)

proba = modelo.predict_proba(X_test)[:,1]

In [ ]:
print("Accuracy")

accuracy_score(y_test,pred)

In [ ]:
print("Recall")

recall_score(y_test,pred)

In [ ]:
print("F1")

f1_score(y_test,pred)

In [ ]:
print("AUC")

roc_auc_score(y_test,proba)

# Estadístico Kolmogorov-Smirnov (KS)

El estadístico KS mide la capacidad del modelo para separar correctamente ambas clases. Cuanto mayor sea el valor, mejor será la discriminación del modelo.

In [ ]:
from scipy.stats import ks_2samp

In [ ]:
ks = ks_2samp(

    proba[y_test==0],

    proba[y_test==1]

)

print("KS =",ks.statistic)

# Matriz de confusión

In [ ]:
ConfusionMatrixDisplay.from_estimator(

    modelo,

    X_test,

    y_test,

    cmap="Blues"

)

plt.show()

# Curva ROC

In [ ]:
fpr,tpr,_ = roc_curve(

    y_test,

    proba

)

plt.figure(figsize=(7,6))

plt.plot(

    fpr,

    tpr,

    label="Modelo Optimizado"

)

plt.plot(

    [0,1],

    [0,1],

    "--"

)

plt.xlabel("False Positive Rate")

plt.ylabel("True Positive Rate")

plt.title("Curva ROC")

plt.legend()

plt.show()

In [ ]:
tabla_final = pd.DataFrame({

"Métrica":[

"Accuracy",

"Precision",

"Recall",

"F1",

"AUC",

"KS"

],

"Resultado":[

accuracy_score(y_test,pred),

precision_score(y_test,pred),

recall_score(y_test,pred),

f1_score(y_test,pred),

roc_auc_score(y_test,proba),

ks.statistic

]

})

tabla_final

# Selección del mejor modelo

Después de comparar los cuatro algoritmos y optimizar el modelo con mejor desempeño, se seleccionó el algoritmo que obtuvo el mayor valor de AUC junto con un equilibrio adecuado entre Accuracy, Precision, Recall y F1-Score.

La optimización mediante GridSearchCV permitió mejorar el rendimiento inicial del modelo, mientras que el estadístico KS confirmó una buena capacidad para discriminar entre viviendas de precio alto y precio bajo.

# Conclusiones

Durante el desarrollo de este proyecto se aplicó un flujo completo de Machine Learning supervisado, desde la exploración de los datos hasta la evaluación y optimización de modelos predictivos.

La transformación de la variable **SalePrice** en una clasificación binaria permitió adaptar el conjunto de datos a los requerimientos de la actividad, haciendo posible el uso de métricas de clasificación como Accuracy, Precision, Recall, F1-Score, AUC y KS.

Tras comparar los algoritmos evaluados, el modelo con mejor desempeño fue seleccionado considerando tanto su capacidad predictiva como su estabilidad durante la validación cruzada.

Este trabajo demuestra la importancia de la preparación de datos, la selección adecuada de algoritmos y la optimización de hiperparámetros para construir modelos de clasificación confiables.

# Bibliografía

- Bishop, C. M. (2006). *Pattern Recognition and Machine Learning*. Springer.
- Breiman, L. (2001). *Random Forests*. Machine Learning, 45(1), 5–32.
- Chen, T., & Guestrin, C. (2016). *XGBoost: A Scalable Tree Boosting System*. KDD.
- Goodfellow, I., Bengio, Y., & Courville, A. (2016). *Deep Learning*. MIT Press.
- Hastie, T., Tibshirani, R., & Friedman, J. (2009). *The Elements of Statistical Learning*. Springer.
- James, G., Witten, D., Hastie, T., & Tibshirani, R. (2021). *An Introduction to Statistical Learning* (2nd ed.). Springer.